# Hybrid Latent-State Language Model — Benchmark

This notebook is a self-contained wrapper around `bench.py`, the reusable
entry point for training, strict exact-match QA evaluation, and reporting.
It recreates `src/` + `bench.py` from embedded cells (so improving `src/`
or `bench.py` and regenerating this notebook keeps them in sync), then runs
bench.py as a subprocess.

**Hypothesis:** `latent_state_update() × N` + `decode_token() × M` beats
token-by-token next-token prediction on long-horizon reasoning, at equal
parameter count.


In [ ]:
import subprocess, sys, os, json, time, random, math
from pathlib import Path

# IMPORTANT: do NOT import torch here. Training runs in a SEPARATE
# subprocess (bench.py), so the notebook process only needs the right
# torch installed in the environment. Importing torch in this process and
# then reloading it after a P100 downgrade causes the
# 'Only a single TORCH_LIBRARY can be used to register triton' error.
# Detect the GPU via nvidia-smi and, if it's a P100 (sm_60), install a
# compatible torch into the env; the bench.py subprocess will load it.

def gpu_capability():
    try:
        out = subprocess.check_output(
            ['nvidia-smi', '--query-gpu=compute_cap', '--format=csv,noheader'],
            stderr=subprocess.DEVNULL).decode().strip().splitlines()
        if out:
            return float(out[0].strip())
    except Exception:
        pass
    return None

cap = gpu_capability()
print('GPU compute capability:', cap)
if cap is not None and cap < 7.0:
    print('P100 (sm_60) detected -> installing torch 2.3.1+cu118...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
        '--index-url', 'https://download.pytorch.org/whl/cu118',
        'torch==2.3.1', 'torchvision==0.18.1'])
else:
    print('Using Kaggle-default torch (T4 / newer GPU).')
print('bench.py will report the torch version it loads.')


In [ ]:
# Recreate the reusable src/ package and bench.py from the embedded cells.
import os
os.makedirs('src', exist_ok=True)
open('src/__init__.py', 'w').close()  # optional, enables `import src...`
print('workspace files ready:', sorted(os.listdir('.')))


In [ ]:
%%writefile src/dataset.py
"""
Toy world generator for synthetic reasoning tasks.

Tasks:
  1. location_tracking - where is X after a long, *interfering* sequence of moves?
  2. inventory_tracking - what does X hold after pickups/drops?
  3. transfer          - multi-hop: where is the item after it changed hands + moved?
  4. exact_recall      - remember a password seen far earlier amid decoys (anti-recency)
  5. story_continuation

Design notes (see AGENTS.md: "too easy / low loss but useless output"):
  * Interfering distractors use REAL entity names and locations, so a model that
    just grabs "the last location word it saw" or attends to the most recent
    mention will be actively misled. This makes low loss require real tracking.
  * Every QA sample is emitted in a strict "Answer: <X>" slot. Evaluation can
    therefore do EXACT MATCH on the generated answer rather than a lazy substring
    check, which is what previously hid useless output behind a low loss.
  * Each sample carries a `meta` dict (difficulty/structure) so evaluation can
    break accuracy down by CONDITION (interference level, recall gap, decoy
    present, multi-hop depth) -- far more informative than a single number.
"""

import random
import json
from dataclasses import dataclass, field
from typing import List, Tuple, Dict, Optional


# --- shared pools ---------------------------------------------------------

FILLER = [
    "It was a quiet day.",
    "The sun was shining.",
    "Time passed slowly.",
    "Nothing unusual happened.",
    "The weather was pleasant.",
    "A soft wind moved through the trees.",
    "The clock on the wall kept ticking.",
    "Shadows stretched long across the floor.",
]

# Literal label used to anchor the answer slot in training and evaluation.
ANSWER_LABEL = "Answer:"


def _interference_sentence(world) -> str:
    """Distractor that mentions a REAL entity/location to create interference."""
    name = random.choice(list(world.entities.keys()))
    loc = random.choice(world.locations)
    templates = [
        f"{name} walked toward the {loc}.",
        f"{name} was seen near the {loc}.",
        f"The {loc} looked different today.",
        f"{name} spent some time in the {loc}.",
        f"Someone mentioned {name} at the {loc}.",
        f"{name} left the {loc} in a hurry.",
    ]
    return random.choice(templates)


@dataclass
class Entity:
    name: str
    location: str
    inventory: List[str] = field(default_factory=list)


@dataclass
class World:
    entities: Dict[str, Entity] = field(default_factory=dict)
    locations: List[str] = field(default_factory=list)
    items: List[str] = field(default_factory=list)
    history: List[str] = field(default_factory=list)
    names: List[str] = field(default_factory=list)

    def __init__(self, names=None, locations=None, items=None):
        self.locations = locations or [
            "kitchen", "bedroom", "garden", "garage", "bathroom",
            "living room", "office", "basement", "attic", "hallway"
        ]
        self.items = items or [
            "apple", "book", "key", "phone", "cup", "pen",
            "wallet", "watch", "bag", "umbrella"
        ]
        self.names = names or [
            "John", "Mary", "Alex", "Sam", "Emma", "Leo",
            "Zoe", "Max", "Lily", "Tom"
        ]
        self.entities = {}
        self.history = []
        self._init_world()

    def _init_world(self):
        names_subset = random.sample(self.names, k=random.randint(3, 5))
        for i, name in enumerate(names_subset):
            loc = random.choice(self.locations)
            if i == 0:
                inv = random.sample(self.items, k=random.randint(1, 2))
            else:
                inv = random.sample(self.items, k=random.randint(0, 2))
            self.entities[name] = Entity(name=name, location=loc, inventory=inv)

    def _narrate(self, sentence: str):
        self.history.append(sentence)

    def move_entity(self, name: str) -> str:
        entity = self.entities[name]
        new_loc = random.choice([l for l in self.locations if l != entity.location])
        old_loc = entity.location
        entity.location = new_loc
        sentence = f"{name} moved from {old_loc} to {new_loc}."
        self._narrate(sentence)
        return sentence

    def pickup_item(self, name: str) -> str:
        entity = self.entities[name]
        available = [i for i in self.items if i not in entity.inventory]
        if not available:
            return ""
        item = random.choice(available)
        entity.inventory.append(item)
        sentence = f"{name} picked up {item}."
        self._narrate(sentence)
        return sentence

    def drop_item(self, name: str) -> str:
        entity = self.entities[name]
        if not entity.inventory:
            return ""
        item = random.choice(entity.inventory)
        entity.inventory.remove(item)
        sentence = f"{name} dropped {item}."
        self._narrate(sentence)
        return sentence

    def give_item(self, name1: str, name2: str) -> str:
        e1 = self.entities[name1]
        e2 = self.entities[name2]
        if not e1.inventory or e1.location != e2.location:
            return ""
        item = random.choice(e1.inventory)
        e1.inventory.remove(item)
        e2.inventory.append(item)
        sentence = f"{name1} gave {item} to {name2}."
        self._narrate(sentence)
        return sentence

    def transfer_item(self, item: str, from_name: str, to_name: str) -> str:
        """Force a transfer (co-locating if needed) and narrate it.

        Used by the multi-hop transfer task where the answer depends on the
        item's final holder's location.
        """
        if item not in self.entities[from_name].inventory:
            return ""
        if self.entities[from_name].location != self.entities[to_name].location:
            self.entities[to_name].location = self.entities[from_name].location
        self.entities[from_name].inventory.remove(item)
        self.entities[to_name].inventory.append(item)
        sentence = f"{from_name} gave the {item} to {to_name}."
        self._narrate(sentence)
        return sentence

    def generate_password(self) -> Tuple[str, str]:
        chars = "ABCDEFGHJKLMNPQRSTUVWXYZ23456789"
        password = "".join(random.choices(chars, k=8))
        entity = random.choice(list(self.entities.keys()))
        sentence = f"The secret code for {entity} is {password}."
        self._narrate(sentence)
        return password, entity

    def _interference(self) -> str:
        return _interference_sentence(self)


def generate_location_task(
    n_moves: int = 10,
    n_interference: int = 8,
    n_filler: int = 3,
    max_chars: int = 600,
) -> Tuple[str, str, str, dict]:
    """Location tracking with interfering mentions of other entities/places."""
    world = World()
    sentences = [
        f"{name} was in the {entity.location}."
        for name, entity in world.entities.items()
    ]

    names = list(world.entities.keys())
    n_interf = 0
    for _ in range(n_moves):
        name = random.choice(names)
        action = random.choice(["move", "pickup", "drop"])
        if action == "move":
            sentences.append(world.move_entity(name))
        elif action == "pickup":
            s = world.pickup_item(name)
            if s:
                sentences.append(s)
        else:
            s = world.drop_item(name)
            if s:
                sentences.append(s)
        if random.random() < 0.5:
            sentences.append(world._interference())
            n_interf += 1
        if len(" ".join(sentences)) > max_chars:
            break

    for _ in range(n_filler):
        sentences.append(random.choice(FILLER))
        if len(" ".join(sentences)) > max_chars:
            break

    narrative = " ".join(s for s in sentences if s)
    target = random.choice(names)
    question = f"Where is {target}?"
    answer = world.entities[target].location
    meta = {"n_moves": n_moves, "n_interference": n_interf}
    return narrative, question, answer, meta


def generate_inventory_task(
    n_actions: int = 6,
    n_interference: int = 4,
    max_chars: int = 600,
) -> Tuple[str, str, str, dict]:
    """Inventory tracking with interfering mentions."""
    world = World()
    sentences = []

    for name, entity in world.entities.items():
        if entity.inventory:
            items_str = ", ".join(entity.inventory)
            sentences.append(f"{name} was in the {entity.location} with {items_str}.")
        else:
            sentences.append(f"{name} was in the {entity.location}.")

    names = list(world.entities.keys())
    n_interf = 0
    for _ in range(n_actions):
        name = random.choice(names)
        action = random.choice(["move", "pickup", "drop"])
        if action == "move":
            sentences.append(world.move_entity(name))
        elif action == "pickup":
            s = world.pickup_item(name)
            if s:
                sentences.append(s)
        else:
            s = world.drop_item(name)
            if s:
                sentences.append(s)
        if random.random() < 0.4:
            sentences.append(world._interference())
            n_interf += 1
        if len(" ".join(sentences)) > max_chars:
            break

    narrative = " ".join(s for s in sentences if s)
    entities_with_items = [n for n in names if world.entities[n].inventory]
    target = random.choice(entities_with_items) if entities_with_items else random.choice(names)
    question = f"What does {target} have?"
    answer = " and ".join(world.entities[target].inventory) if world.entities[target].inventory else "nothing"
    meta = {"n_actions": n_actions, "n_interference": n_interf}
    return narrative, question, answer, meta


def generate_transfer_task(
    n_steps: int = 7,
    n_interference: int = 6,
    max_chars: int = 600,
) -> Tuple[str, str, str, dict]:
    """Multi-hop: track an item as it changes hands and its holder moves.

    The answer (the item's final location) requires combining two reasoning
    steps: who currently holds the item, and where that holder ended up.
    """
    world = World()
    sentences = [
        f"{name} was in the {entity.location}."
        for name, entity in world.entities.items()
    ]

    names = list(world.entities.keys())
    item = random.choice(world.items)
    holder = random.choice(names)
    world.entities[holder].inventory.append(item)
    sentences.append(f"{holder} had the {item}.")

    n_interf = 0
    for _ in range(n_steps):
        r = random.random()
        if r < 0.45:
            recip = random.choice([n for n in names if n != holder])
            s = world.transfer_item(item, holder, recip)
            if s:
                sentences.append(s)
                holder = recip
            else:
                sentences.append(world.move_entity(holder))
        else:
            sentences.append(world.move_entity(holder))
        if random.random() < 0.5:
            sentences.append(world._interference())
            n_interf += 1
        if len(" ".join(sentences)) > max_chars:
            break

    narrative = " ".join(s for s in sentences if s)
    question = f"Where is the {item}?"
    answer = world.entities[holder].location
    meta = {"n_hops": n_steps, "n_interference": n_interf}
    return narrative, question, answer, meta


def generate_recall_task(
    n_distractor_sentences: int = 40,
    decoy_for_target: bool = True,
    max_chars: int = 600,
) -> Tuple[str, str, str, dict]:
    """Exact recall with a long gap, decoy codes, and an anti-recency trap.

    The target code is shown early. The distractor stream contains many other
    codes (for other entities) plus interfering filler. If decoy_for_target is
    set, the SAME entity's code is later "updated", and the question asks for
    the FIRST code -- so a model relying on recency will be wrong.

    Distractors are added until the narrative reaches `max_chars`, which keeps
    the whole sample (narrative + question + answer) inside a small char-level
    context window instead of being silently truncated.
    """
    world = World()
    sentences = [
        f"{name} was in the {entity.location}."
        for name, entity in world.entities.items()
    ]

    password, entity_name = world.generate_password()
    sentences.append(f"The secret code for {entity_name} is {password}.")

    other_entities = [n for n in world.entities if n != entity_name]
    n_added = 0
    for _ in range(n_distractor_sentences):
        r = random.random()
        if r < 0.3 and other_entities:
            p2, e2 = world.generate_password()
            sentences.append(f"The secret code for {e2} is {p2}.")
        elif r < 0.6:
            sentences.append(world._interference())
        else:
            sentences.append(random.choice(FILLER))
        n_added += 1
        if len(" ".join(sentences)) > max_chars:
            break

    gap_chars = len(" ".join(sentences))  # distance from code to question

    if decoy_for_target:
        chars = "ABCDEFGHJKLMNPQRSTUVWXYZ23456789"
        decoy = "".join(random.choices(chars, k=8))
        sentences.append(f"The secret code for {entity_name} was updated to {decoy}.")
        question = f"What was the FIRST secret code for {entity_name}?"
    else:
        question = f"What is the secret code for {entity_name}?"

    answer = password
    narrative = " ".join(s for s in sentences if s)
    meta = {
        "n_distractors": n_added,
        "gap_chars": gap_chars,
        "has_decoy": decoy_for_target,
    }
    return narrative, question, answer, meta


def generate_story_prompt(
    n_setup_sentences: int = 3,
) -> Tuple[str, str, dict]:
    """Story continuation prompt + ground-truth opening line."""
    world = World()
    sentences = []

    name = list(world.entities.keys())[0]
    entity = world.entities[name]

    sentences.append(f"Write a story about {name}.")
    sentences.append(f"{name} was in the {entity.location}.")

    for _ in range(n_setup_sentences - 1):
        actions = [
            f"{name} looked around.",
            f"{name} felt something was about to happen.",
            f"Something caught {name}'s attention.",
            f"{name} heard a strange noise.",
            f"The air felt different.",
        ]
        sentences.append(random.choice(actions))

    prompt = " ".join(sentences)

    continuations = [
        f"Suddenly, {name} noticed something unusual.",
        f"Then {name} decided to explore further.",
        f"Out of nowhere, a mysterious figure appeared.",
        f"At that moment, everything changed.",
    ]
    ground_truth = random.choice(continuations)
    meta = {}
    return prompt, ground_truth, meta


# --- formatting helpers ---------------------------------------------------
# These guarantee train/eval use the SAME surface form so the model learns to
# emit its answer in the "Answer:" slot, enabling strict exact-match scoring.

def format_for_training(sample: dict) -> str:
    """Full text the model is trained on (narrative + question + answer)."""
    if sample["task_type"] == "story":
        return f"{sample['narrative']}\n{sample['answer']}"
    return (
        f"{sample['narrative']}\n"
        f"Question: {sample['question']}\n"
        f"{ANSWER_LABEL} {sample['answer']}"
    )


def build_prompt(sample: dict) -> str:
    """Prompt fed at inference: narrative + question + 'Answer: '.

    Ends with a trailing space so the surface form exactly matches
    format_for_training (which writes 'Answer: <ans>'); otherwise the model
    sees a context at inference it never encountered during training.
    """
    if sample["task_type"] == "story":
        return sample["narrative"] + "\n"
    return (
        f"{sample['narrative']}\n"
        f"Question: {sample['question']}\n"
        f"{ANSWER_LABEL} "
    )


def parse_answer(generated: str) -> str:
    """Extract the predicted answer from generated text (before any newline)."""
    text = generated.strip()
    if "\n" in text:
        text = text.split("\n", 1)[0].strip()
    if text.lower().startswith(ANSWER_LABEL.lower()):
        text = text[len(ANSWER_LABEL):].strip()
    return text


def _bucket(sample: dict) -> str:
    """Coarse difficulty bucket for stratified accuracy reporting."""
    t = sample["task_type"]
    m = sample.get("meta", {})
    if t in ("location", "inventory", "transfer"):
        ni = m.get("n_interference", 0)
        if ni == 0:
            return "interf=0"
        if ni <= 2:
            return "interf=1-2"
        return "interf>=3"
    if t == "recall":
        if m.get("has_decoy"):
            return "decoy=yes"
        return "decoy=no"
    return "all"


def generate_dataset(
    n_samples: int = 1000,
    seed: int = 42,
    task_weights: Optional[Dict[str, float]] = None,
    location_max_chars: int = 600,
    inventory_max_chars: int = 600,
    transfer_max_chars: int = 600,
    recall_max_chars: int = 600,
) -> List[Dict]:
    """Generate a mixed dataset of reasoning tasks.

    `*-max-chars` controls narrative length per task type (so the longest
    sample still fits the model's context window). For a quick CPU run you can
    pass smaller values (e.g. recall_max_chars=400) to keep context short.
    """
    random.seed(seed)

    if task_weights is None:
        task_weights = {
            "location": 0.3,
            "inventory": 0.2,
            "transfer": 0.2,
            "recall": 0.2,
            "story": 0.1,
        }

    tasks = list(task_weights.keys())
    weights = list(task_weights.values())

    dataset = []
    for _ in range(n_samples):
        task_type = random.choices(tasks, weights=weights, k=1)[0]

        if task_type == "location":
            narrative, question, answer, meta = generate_location_task(max_chars=location_max_chars)
        elif task_type == "inventory":
            narrative, question, answer, meta = generate_inventory_task(max_chars=inventory_max_chars)
        elif task_type == "transfer":
            narrative, question, answer, meta = generate_transfer_task(max_chars=transfer_max_chars)
        elif task_type == "recall":
            narrative, question, answer, meta = generate_recall_task(max_chars=recall_max_chars)
        elif task_type == "story":
            narrative, answer, meta = generate_story_prompt()
            question = ""
        else:
            continue

        dataset.append({
            "narrative": narrative,
            "question": question,
            "answer": answer,
            "task_type": task_type,
            "meta": meta,
        })

    return dataset


if __name__ == "__main__":
    dataset = generate_dataset(n_samples=8, seed=42)
    for i, sample in enumerate(dataset):
        print(f"\n=== Sample {i+1} [{sample['task_type']}] meta={sample['meta']} ===")
        print(f"Narrative: {sample['narrative']}")
        if sample["question"]:
            print(f"Question: {sample['question']}")
        print(f"Answer: {sample['answer']}")
        print(f"-- train text --\n{format_for_training(sample)}")


In [ ]:
%%writefile src/tokenizer.py
"""
Simple character-level tokenizer for synthetic tasks.

Since we're generating our own text, we don't need a heavy tokenizer.
Character-level is sufficient for the toy world tasks and keeps vocab small.
"""

from collections import Counter
from typing import List, Dict, Optional, Tuple
import json
import os


class CharTokenizer:
    """Character-level tokenizer with special tokens."""

    def __init__(
        self,
        texts: Optional[List[str]] = None,
        max_vocab: int = 256,
        pad_token: str = "<PAD>",
        unk_token: str = "<UNK>",
        eos_token: str = "<EOS>",
        sep_token: str = "<SEP>",
    ):
        self.pad_token = pad_token
        self.unk_token = unk_token
        self.eos_token = eos_token
        self.sep_token = sep_token

        # Build vocabulary
        self.char_counts = Counter()
        if texts:
            for text in texts:
                self.char_counts.update(text)

        # Special tokens first
        self.special_tokens = [pad_token, unk_token, eos_token, sep_token]
        self.vocab = {c: idx + len(self.special_tokens) for idx, (c, _) in enumerate(self.char_counts.most_common(max_vocab - len(self.special_tokens)))}
        for i, tok in enumerate(self.special_tokens):
            self.vocab[tok] = i

        self.inv_vocab = {i: c for c, i in self.vocab.items()}
        self.vocab_size = len(self.vocab)

    def encode(self, text: str, max_len: Optional[int] = None) -> List[int]:
        """Encode text to token IDs."""
        ids = [self.vocab.get(c, self.vocab[self.unk_token]) for c in text]
        if max_len:
            ids = ids[:max_len]
        return ids

    def decode(self, ids: List[int]) -> str:
        """Decode token IDs to text."""
        return "".join(self.inv_vocab.get(i, self.unk_token) for i in ids)

    def save(self, path: str):
        """Save vocabulary to JSON."""
        os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
        with open(path, 'w') as f:
            json.dump({
                "vocab": self.vocab,
                "special_tokens": self.special_tokens,
                "vocab_size": self.vocab_size,
            }, f)

    @classmethod
    def load(cls, path: str) -> "CharTokenizer":
        """Load vocabulary from JSON."""
        with open(path) as f:
            data = json.load(f)
        tok = cls()
        tok.vocab = data["vocab"]
        tok.special_tokens = data["special_tokens"]
        tok.inv_vocab = {i: c for c, i in tok.vocab.items()}
        tok.vocab_size = len(tok.vocab)
        return tok


def build_tokenizer_from_dataset(dataset: List[dict], max_vocab: int = 256) -> CharTokenizer:
    """Build tokenizer from dataset samples.

    Includes the QA formatting markers (Question:, Answer:, newline, ':') so
    they are never mapped to <UNK> when the strict "Answer: <X>" surface form
    is used during training/evaluation.
    """
    texts = []
    for sample in dataset:
        texts.append(sample["narrative"])
        if sample.get("question"):
            texts.append(sample["question"])
        texts.append(sample["answer"])
    # Ensure format markers are present in the vocabulary.
    texts += ["Question:", "Answer:", "\n", ":", "The secret code for was updated to"]
    return CharTokenizer(texts, max_vocab=max_vocab)


In [ ]:
%%writefile src/models.py
"""
Model architectures for the hybrid latent-state language model.

Models:
  - BaselineTransformer: standard autoregressive transformer
  - LatentSSM: recurrent latent model with thinking loop (causal LM)
  - LatentSSMDecoder: SSM + FFN decoder (cheap multi-token generation)

Key design: All models produce [batch, seq_len, vocab_size] output for
fair comparison on next-token prediction.

The latent models additionally perform N "thinking" steps in latent space
between processing input tokens, testing whether extra latent computation
improves reasoning over pure token-by-token processing.
"""

import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Optional, Tuple


# ============================================================
# Baseline Transformer
# ============================================================

class PositionalEncoding(nn.Module):
    """Sinusoidal positional encoding, generated on the fly for any length."""
    def __init__(self, d_model: int, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        self.d_model = d_model
        self._cache: dict = {}

    def forward(self, x):
        seq_len = x.size(1)
        device = x.device
        if seq_len not in self._cache or self._cache[seq_len].device != device:
            position = torch.arange(0, seq_len, dtype=torch.float, device=device).unsqueeze(1)
            div_term = torch.exp(
                torch.arange(0, self.d_model, 2, dtype=torch.float, device=device)
                * (-math.log(10000.0) / self.d_model)
            )
            pe = torch.zeros(seq_len, self.d_model, device=device)
            pe[:, 0::2] = torch.sin(position * div_term)
            pe[:, 1::2] = torch.cos(position * div_term)
            self._cache[seq_len] = pe.unsqueeze(0)
        return self.dropout(x + self._cache[seq_len])


class BaselineTransformer(nn.Module):
    """
    Standard autoregressive transformer baseline.

    Architecture:
      tokens -> embedding + positional encoding -> transformer layers -> logits

    Output: [batch, seq_len, vocab_size]
    """

    def __init__(
        self,
        vocab_size: int,
        d_model: int = 256,
        nhead: int = 8,
        num_layers: int = 4,
        dim_ff: int = 512,
        dropout: float = 0.1,
        max_seq_len: int = 512,
    ):
        super().__init__()
        self.d_model = d_model
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoder = PositionalEncoding(d_model, dropout)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_ff,
            dropout=dropout,
            batch_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.output_proj = nn.Linear(d_model, vocab_size)

    def forward(self, src: torch.Tensor) -> torch.Tensor:
        """
        Args:
            src: [batch, seq_len] token IDs
        Returns:
            logits: [batch, seq_len, vocab_size]
        """
        mask = self._generate_square_subsequent_mask(src.size(1), src.device)
        x = self.embedding(src) * math.sqrt(self.d_model)
        x = self.pos_encoder(x)
        x = self.transformer(x, mask=mask)
        logits = self.output_proj(x)
        return logits

    @staticmethod
    def _generate_square_subsequent_mask(sz: int, device: torch.device) -> torch.Tensor:
        mask = torch.triu(torch.ones(sz, sz, device=device), diagonal=1)
        mask = mask.masked_fill(mask == 1, float('-inf'))
        return mask


# ============================================================
# SSM Layer (simplified Mamba-style)
# ============================================================

class SSMLayer(nn.Module):
    """
    State Space Model layer with input-dependent dynamics (Mamba-style).

    Uses a recurrent update: h_t = A(x_t) * h_{t-1} + B(x_t) * x_t
    where A and B are functions of the input (selective mechanism).

    This allows the model to selectively remember/forget based on input.
    """

    def __init__(self, d_state: int, d_input: int, selective: bool = True):
        super().__init__()
        self.d_state = d_state
        self.d_input = d_input
        self.selective = selective

        # Base state transition matrix
        self.A_base = nn.Parameter(torch.randn(d_state, d_state) * 0.1)
        
        if selective:
            # Input-dependent modulation of A
            self.A_mod = nn.Linear(d_input, d_state * d_state, bias=False)
            # Input-dependent B
            self.B = nn.Linear(d_input, d_state, bias=False)
        else:
            # Simple fixed A and B
            self.A = self.A_base
            self.B = nn.Linear(d_input, d_state)

        # Output projection
        self.C = nn.Linear(d_state, d_state)

        # Nonlinearity
        self.act = nn.SiLU()

        # Normalization
        self.norm = nn.LayerNorm(d_state)

    def forward(self, x: torch.Tensor, h: Optional[torch.Tensor] = None) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Process a sequence of inputs.

        Args:
            x: [batch, seq_len, d_input] input
            h: [batch, d_state] initial state (optional)
        Returns:
            output: [batch, seq_len, d_state]
            h_final: [batch, d_state] final state
        """
        batch, seq_len, _ = x.shape
        if h is None:
            h = torch.zeros(batch, self.d_state, device=x.device)

        outputs = []
        for t in range(seq_len):
            h = self.step(x[:, t, :], h)
            outputs.append(h)

        outputs = torch.stack(outputs, dim=1)  # [batch, seq_len, d_state]
        return outputs, h

    def step(self, x: torch.Tensor, h: torch.Tensor) -> torch.Tensor:
        """
        Single recurrent step with optional input-dependent dynamics.

        Args:
            x: [batch, d_input] input at current step
            h: [batch, d_state] current state
        Returns:
            h_new: [batch, d_state] updated state
        """
        if self.selective:
            # Compute input-dependent A: A(x) = A_base + A_mod(x)
            batch_size = x.size(0)
            A_delta = self.A_mod(x).view(batch_size, self.d_state, self.d_state)
            # Small modulation to keep stable
            A = self.A_base.unsqueeze(0) + 0.1 * torch.tanh(A_delta)
            # State update: h_new = A @ h + B(x)
            h = torch.bmm(A, h.unsqueeze(-1)).squeeze(-1) + self.B(x)
        else:
            # Simple linear update
            h = torch.einsum('ij,bj->bi', self.A_base, h) + self.B(x)
        
        h = self.act(h)
        h = self.norm(h)
        return h


# ============================================================
# Latent SSM Model (Causal LM with thinking steps)
# ============================================================

class LatentSSM(nn.Module):
    """
    Recurrent latent model with thinking loop.

    Architecture:
      For each input token:
        1. Embed token
        2. Update SSM state
        3. Every `think_every` tokens: do N latent thinking steps
        4. Decode state -> logits for this position

    This produces [batch, seq_len, vocab_size] output, same as the baseline,
    enabling fair comparison. The latent model does extra computation
    (thinking steps) between token predictions.

    Output: [batch, seq_len, vocab_size]
    """

    def __init__(
        self,
        vocab_size: int,
        d_state: int = 256,
        d_model: int = 256,
        num_ssm_layers: int = 2,
        latent_steps: int = 4,
        think_every: int = 1,
        dropout: float = 0.1,
        selective: bool = True,
    ):
        super().__init__()
        self.d_state = d_state
        self.d_model = d_model
        self.latent_steps = latent_steps
        self.think_every = think_every

        self.embedding = nn.Embedding(vocab_size, d_model)

        # Input projection (token embedding -> SSM input)
        self.input_proj = nn.Linear(d_model, d_state)

        # Stack of SSM layers
        self.ssm_layers = nn.ModuleList([
            SSMLayer(d_state, d_state, selective=selective) for _ in range(num_ssm_layers)
        ])

        # FFN for latent thinking
        self.ffn = nn.Sequential(
            nn.Linear(d_state, d_state * 2),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(d_state * 2, d_state),
        )

        # Token decoder (from state to vocab)
        self.token_decoder = nn.Linear(d_state, vocab_size)

        # Residual gating for thinking steps
        self.gate = nn.Sequential(
            nn.Linear(d_state, d_state),
            nn.Sigmoid(),
        )

    def think(self, h: torch.Tensor, steps: Optional[int] = None) -> torch.Tensor:
        """
        Run latent thinking steps without consuming input tokens.

        Args:
            h: [batch, d_state] current state
            steps: number of thinking steps (default: self.latent_steps)
        Returns:
            h: [batch, d_state] updated state
        """
        steps = steps or self.latent_steps
        for _ in range(steps):
            h_new = h
            for ssm in self.ssm_layers:
                # Self-recurrence: use state as both input and state
                h_new = ssm.step(h_new, h_new)
            h_new = self.ffn(h_new)
            # Gated residual
            gate = self.gate(h)
            h = gate * h_new + (1 - gate) * h
        return h

    def forward(self, src: torch.Tensor) -> torch.Tensor:
        """
        Full forward pass: process tokens sequentially with optional thinking.

        Args:
            src: [batch, seq_len] token IDs
        Returns:
            logits: [batch, seq_len, vocab_size]
        """
        batch, seq_len = src.shape
        device = src.device

        # Initialize state
        h = torch.zeros(batch, self.d_state, device=device)

        # Embed all tokens
        x = self.embedding(src)  # [batch, seq, d_model]
        x = self.input_proj(x)   # [batch, seq, d_state]

        all_logits = []

        for t in range(seq_len):
            # Update SSM state with current token
            for ssm in self.ssm_layers:
                h = ssm.step(x[:, t, :], h)

            # Periodic latent thinking
            if self.think_every > 0 and (t + 1) % self.think_every == 0:
                h = self.think(h)

            # Decode to logits
            logits_t = self.token_decoder(h)  # [batch, vocab_size]
            all_logits.append(logits_t)

        logits = torch.stack(all_logits, dim=1)  # [batch, seq_len, vocab_size]
        return logits


# ============================================================
# Latent SSM + Decoder (cheap multi-token generation)
# ============================================================

class LatentSSMDecoder(nn.Module):
    """
    SSM + FFN decoder for multi-token generation.

    Key difference from LatentSSM:
    - After thinking, can generate multiple tokens cheaply from same state
    - Decoder uses positional embeddings to generate tokens_per_step tokens
    - State evolves slowly, tokens generated quickly

    Training mode: produces [batch, seq_len, vocab_size] for fair comparison
    Generation mode: can produce multiple tokens per state update

    Output: [batch, seq_len, vocab_size]
    """

    def __init__(
        self,
        vocab_size: int,
        d_state: int = 256,
        d_model: int = 256,
        num_ssm_layers: int = 2,
        latent_steps: int = 4,
        tokens_per_step: int = 8,
        think_every: int = 4,
        dropout: float = 0.1,
        selective: bool = True,
    ):
        super().__init__()
        self.d_state = d_state
        self.d_model = d_model
        self.latent_steps = latent_steps
        self.tokens_per_step = tokens_per_step
        self.think_every = think_every

        self.embedding = nn.Embedding(vocab_size, d_model)
        self.input_proj = nn.Linear(d_model, d_state)

        # SSM layers
        self.ssm_layers = nn.ModuleList([
            SSMLayer(d_state, d_state, selective=selective) for _ in range(num_ssm_layers)
        ])

        # FFN for latent thinking
        self.ffn = nn.Sequential(
            nn.Linear(d_state, d_state * 2),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(d_state * 2, d_state),
        )

        # Token decoder with position encoding
        self.decoder_ffn = nn.Sequential(
            nn.Linear(d_state + 32, d_state),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(d_state, vocab_size),
        )
        self.pos_embed = nn.Parameter(torch.randn(tokens_per_step, 32))

        # Gating
        self.gate = nn.Sequential(
            nn.Linear(d_state, d_state),
            nn.Sigmoid(),
        )

    def think(self, h: torch.Tensor, steps: Optional[int] = None) -> torch.Tensor:
        """Run latent thinking steps."""
        steps = steps or self.latent_steps
        for _ in range(steps):
            h_new = h
            for ssm in self.ssm_layers:
                h_new = ssm.step(h_new, h_new)
            h_new = self.ffn(h_new)
            gate = self.gate(h)
            h = gate * h_new + (1 - gate) * h
        return h

    def decode_tokens(self, h: torch.Tensor, n_tokens: int) -> torch.Tensor:
        """
        Generate multiple tokens from a single state.

        Args:
            h: [batch, d_state] state
            n_tokens: number of tokens to generate
        Returns:
            logits: [batch, n_tokens, vocab_size]
        """
        n_tokens = min(n_tokens, self.tokens_per_step)
        batch = h.size(0)

        h_expanded = h.unsqueeze(1).expand(-1, n_tokens, -1)
        pos = self.pos_embed[:n_tokens].unsqueeze(0).expand(batch, -1, -1)

        x = torch.cat([h_expanded, pos], dim=-1)
        logits = self.decoder_ffn(x)
        return logits

    def forward(self, src: torch.Tensor) -> torch.Tensor:
        """
        Full forward pass for training.

        Process tokens sequentially, periodically think, and decode
        multiple tokens per state. For fair comparison with baseline,
        produces [batch, seq_len, vocab_size].

        Args:
            src: [batch, seq_len] token IDs
        Returns:
            logits: [batch, seq_len, vocab_size]
        """
        batch, seq_len = src.shape
        device = src.device

        # Initialize state
        h = torch.zeros(batch, self.d_state, device=device)

        # Embed all tokens
        x = self.embedding(src)
        x = self.input_proj(x)

        all_logits = []
        token_idx = 0  # How many output tokens we've produced from current state

        for t in range(seq_len):
            # Update SSM state with current token
            for ssm in self.ssm_layers:
                h = ssm.step(x[:, t, :], h)

            # Periodic latent thinking
            if self.think_every > 0 and (t + 1) % self.think_every == 0:
                h = self.think(h)
                token_idx = 0  # Reset token counter after thinking

            # Decode from current state
            if token_idx < self.tokens_per_step:
                logits_t = self.decode_tokens(h, 1)  # [batch, 1, vocab]
                logits_t = logits_t.squeeze(1)  # [batch, vocab]
                token_idx += 1
            else:
                # Just decode directly without positional bias
                logits_t = self.decode_tokens(h, 1).squeeze(1)

            all_logits.append(logits_t)

        logits = torch.stack(all_logits, dim=1)  # [batch, seq_len, vocab_size]
        return logits


In [ ]:
%%writefile src/trainer.py
"""
Training loop for latent-state language model experiments.

Handles:
  - Training with multiple loss types
  - Experiment tracking
  - Checkpointing
  - Evaluation
  - Sample generation
"""

import os
import json
import time
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Tuple
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

try:
    from src.dataset import build_prompt, parse_answer, _bucket
except ImportError:  # allow running trainer standalone
    from dataset import build_prompt, parse_answer, _bucket


@dataclass
class ExperimentConfig:
    """Configuration for a single experiment."""
    exp_id: str = "exp001"
    model: str = "baseline"  # baseline, latent_ssm, latent_ssm_decoder

    # Model params
    d_model: int = 256
    d_state: int = 256
    nhead: int = 8
    num_layers: int = 4
    num_ssm_layers: int = 2
    latent_steps: int = 4
    tokens_per_step: int = 8

    # Training params
    batch_size: int = 32
    learning_rate: float = 3e-4
    num_epochs: int = 20
    max_seq_len: int = 256
    gradient_clip: float = 1.0
    warmup_steps: int = 100

    # Loss weights
    token_loss_weight: float = 1.0
    latent_consistency_weight: float = 0.1
    state_evolution_weight: float = 0.1

    # Misc
    seed: int = 42
    device: str = "auto"
    save_every: int = 5
    eval_every: int = 1

    def to_dict(self) -> dict:
        return {k: v for k, v in self.__dict__.items()}

    @classmethod
    def from_dict(cls, d: dict) -> "ExperimentConfig":
        return cls(**{k: v for k, v in d.items() if k in cls.__dataclass_fields__})


class TextDataset(Dataset):
    """Simple text dataset for training."""

    def __init__(self, texts: List[str], tokenizer, max_len: int = 256):
        self.texts = texts
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        ids = self.tokenizer.encode(text, max_len=self.max_len)
        ids = ids + [self.tokenizer.vocab[self.tokenizer.eos_token]]
        ids = ids[:self.max_len + 1]

        # Pad to max_len + 1
        while len(ids) < self.max_len + 1:
            ids.append(self.tokenizer.vocab[self.tokenizer.pad_token])

        x = torch.tensor(ids[:-1], dtype=torch.long)
        y = torch.tensor(ids[1:], dtype=torch.long)
        return x, y


class Trainer:
    """Training loop with experiment tracking and checkpointing."""

    def __init__(self, model: nn.Module, config: ExperimentConfig, tokenizer, output_dir: str = "experiments"):
        self.model = model
        self.config = config
        self.tokenizer = tokenizer
        self.output_dir = Path(output_dir) / config.exp_id
        self.output_dir.mkdir(parents=True, exist_ok=True)

        # Device
        if config.device == "auto":
            self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        else:
            self.device = torch.device(config.device)
        self.model = self.model.to(self.device)

        # Optimizer
        self.optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=config.learning_rate,
            weight_decay=0.01,
        )

        # Scheduler
        self.scheduler = torch.optim.lr_scheduler.OneCycleLR(
            self.optimizer,
            max_lr=config.learning_rate,
            total_steps=config.num_epochs * 100,  # approximate
            pct_start=0.1,
        )

        # Metrics
        self.metrics = {
            "train_loss": [],
            "val_loss": [],
            "eval_accuracy": [],
            "samples": [],
        }

    def train_epoch(self, dataloader: DataLoader) -> float:
        """Run one training epoch."""
        self.model.train()
        total_loss = 0
        n_batches = 0

        pbar = tqdm(dataloader, desc=f"Training [{self.config.exp_id}]")
        for batch_x, batch_y in pbar:
            batch_x = batch_x.to(self.device)
            batch_y = batch_y.to(self.device)

            self.optimizer.zero_grad()

            # Forward pass - all models produce [batch, seq, vocab]
            logits = self.model(batch_x)
            
            # Reshape for loss computation
            logits = logits.reshape(-1, logits.size(-1))
            targets = batch_y.reshape(-1)
            
            loss = F.cross_entropy(logits, targets, ignore_index=self.tokenizer.vocab[self.tokenizer.pad_token])

            loss.backward()
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.config.gradient_clip)
            self.optimizer.step()
            self.scheduler.step()

            total_loss += loss.item()
            n_batches += 1
            pbar.set_postfix({"loss": f"{loss.item():.4f}"})

        return total_loss / max(n_batches, 1)

    @torch.no_grad()
    def evaluate(self, dataloader: DataLoader) -> float:
        """Run evaluation."""
        self.model.eval()
        total_loss = 0
        n_batches = 0

        for batch_x, batch_y in dataloader:
            batch_x = batch_x.to(self.device)
            batch_y = batch_y.to(self.device)

            logits = self.model(batch_x)
            logits = logits.reshape(-1, logits.size(-1))
            targets = batch_y.reshape(-1)
            
            loss = F.cross_entropy(logits, targets, ignore_index=self.tokenizer.vocab[self.tokenizer.pad_token])
            total_loss += loss.item()
            n_batches += 1

        return total_loss / max(n_batches, 1)

    @torch.no_grad()
    def generate(self, prompt: str, max_tokens: int = 50, temperature: float = 0.8, top_k: int = 40) -> str:
        """Generate text from a prompt.

        temperature <= 0 disables sampling and uses greedy argmax (used for
        deterministic, exact-match evaluation). Generation stops at EOS or a
        newline so we capture just the answer in the "Answer:" slot.
        """
        self.model.eval()

        prompt_ids = self.tokenizer.encode(prompt, max_len=self.config.max_seq_len)
        generated = list(prompt_ids)

        eos_id = self.tokenizer.vocab[self.tokenizer.eos_token]
        newline_id = self.tokenizer.vocab.get("\n", None)

        for _ in range(max_tokens):
            x = torch.tensor([generated], dtype=torch.long).to(self.device)
            logits = self.model(x)
            next_logits = logits[0, -1, :]

            if temperature <= 0:
                next_token = int(torch.argmax(next_logits).item())
            else:
                next_logits = next_logits / temperature
                if top_k > 0:
                    kth = torch.topk(next_logits, top_k)[0][..., -1, None]
                    next_logits[next_logits < kth] = float('-inf')
                probs = F.softmax(next_logits, dim=-1)
                next_token = int(torch.multinomial(probs, num_samples=1).item())

            if next_token == eos_id:
                break
            if newline_id is not None and next_token == newline_id:
                break
            generated.append(next_token)

        return self.tokenizer.decode(generated[len(prompt_ids):])

    @torch.no_grad()
    def run_qa_eval(self, dataset: List[dict], max_new_tokens: int = 20) -> Dict[str, float]:
        """Strict exact-match evaluation on QA tasks, with stratified breakdowns.

        Returns overall accuracy, per-task accuracy, and accuracy stratified by
        difficulty bucket (interference level / decoy presence / etc.) so we can
        see *where* the model fails -- not just a single number. This also
        exposes "low loss but useless output" that the old substring check hid.
        """
        self.model.eval()
        correct = 0
        total = 0
        by_task = {}
        by_bucket = {}

        for sample in dataset:
            if not sample.get("question"):  # skip story tasks
                continue

            task_type = sample["task_type"]
            by_task.setdefault(task_type, {"correct": 0, "total": 0})
            bucket = _bucket(sample)
            bkey = f"{task_type}|{bucket}"
            by_bucket.setdefault(bkey, {"correct": 0, "total": 0})

            prompt = build_prompt(sample)
            expected = sample["answer"].strip().lower()

            generated = self.generate(prompt, max_tokens=max_new_tokens, temperature=0.0)
            predicted = parse_answer(generated).strip().lower()

            ok = (predicted == expected)
            if ok:
                correct += 1
                by_task[task_type]["correct"] += 1
                by_bucket[bkey]["correct"] += 1
            total += 1
            by_task[task_type]["total"] += 1
            by_bucket[bkey]["total"] += 1

        task_accuracy = {
            task: counts["correct"] / max(counts["total"], 1)
            for task, counts in by_task.items()
        }
        stratified = {
            bkey: counts["correct"] / max(counts["total"], 1)
            for bkey, counts in by_bucket.items()
        }
        return {
            "overall_accuracy": correct / max(total, 1),
            "task_accuracy": task_accuracy,
            "stratified": stratified,
            "n": total,
        }

    def save_checkpoint(self, epoch: int, metrics: dict):
        """Save model checkpoint and metrics."""
        checkpoint_path = self.output_dir / f"model_epoch_{epoch}.pt"
        torch.save({
            "epoch": epoch,
            "model_state_dict": self.model.state_dict(),
            "optimizer_state_dict": self.optimizer.state_dict(),
            "metrics": metrics,
            "config": self.config.to_dict(),
        }, checkpoint_path)

        # Save latest as best_model.pt
        latest_path = self.output_dir / "best_model.pt"
        torch.save({
            "epoch": epoch,
            "model_state_dict": self.model.state_dict(),
            "optimizer_state_dict": self.optimizer.state_dict(),
            "metrics": metrics,
            "config": self.config.to_dict(),
        }, latest_path)

    def save_results(self):
        """Save final results."""
        # Save metrics
        metrics_path = self.output_dir / "metrics.json"
        with open(metrics_path, 'w') as f:
            json.dump(self.metrics, f, indent=2)

        # Save config
        config_path = self.output_dir / "config.json"
        with open(config_path, 'w') as f:
            json.dump(self.config.to_dict(), f, indent=2)

    def train(
        self,
        train_loader: DataLoader,
        val_loader: Optional[DataLoader] = None,
        qa_dataset: Optional[List[dict]] = None,
    ):
        """Full training loop."""
        print(f"\n{'='*60}")
        print(f"Experiment: {self.config.exp_id}")
        print(f"Model: {self.config.model}")
        print(f"Device: {self.device}")
        print(f"Parameters: {sum(p.numel() for p in self.model.parameters()):,}")
        print(f"{'='*60}\n")

        start_time = time.time()

        for epoch in range(1, self.config.num_epochs + 1):
            epoch_start = time.time()

            # Train
            train_loss = self.train_epoch(train_loader)
            self.metrics["train_loss"].append(train_loss)

            # Validate
            val_loss = None
            if val_loader:
                val_loss = self.evaluate(val_loader)
                self.metrics["val_loss"].append(val_loss)

            # QA evaluation
            qa_results = None
            if qa_dataset and epoch % self.config.eval_every == 0:
                qa_results = self.run_qa_eval(qa_dataset)
                self.metrics["eval_accuracy"].append(qa_results)

                # Print + record sample outputs so we can SEE whether the
                # model's output is actually useful, not just low-loss.
                print(f"\n--- SAMPLE OUTPUTS (epoch {epoch}) ---")
                for sample in qa_dataset[:4]:
                    if not sample.get("question"):
                        continue
                    prompt = build_prompt(sample)
                    generated = self.generate(prompt, max_tokens=20, temperature=0.0)
                    predicted = parse_answer(generated).strip().lower()
                    expected = sample["answer"].strip().lower()
                    ok = "✓" if predicted == expected else "✗"
                    print(f"  [{ok}] ({sample['task_type']}) Q: {sample['question']}")
                    print(f"       expected: {sample['answer']!r}  got: {generated!r}")
                    self.metrics["samples"].append({
                        "epoch": epoch,
                        "task_type": sample["task_type"],
                        "prompt": prompt,
                        "expected": sample["answer"],
                        "generated": generated,
                        "correct": predicted == expected,
                    })

            epoch_time = time.time() - epoch_start

            # Log
            log_msg = f"Epoch {epoch}/{self.config.num_epochs} | "
            log_msg += f"train_loss: {train_loss:.4f} | "
            if val_loss is not None:
                log_msg += f"val_loss: {val_loss:.4f} | "
            if qa_results:
                log_msg += f"qa_acc: {qa_results['overall_accuracy']:.3f} | "
            log_msg += f"time: {epoch_time:.1f}s"
            print(log_msg)
            print(f"STAGE: train exp={self.config.exp_id} ep={epoch}/{self.config.num_epochs} "
                  f"val_loss={val_loss if val_loss is not None else '-'} "
                  f"qa_acc={qa_results['overall_accuracy'] if qa_results else '-'}")

            # Save checkpoint
            if epoch % self.config.save_every == 0 or epoch == self.config.num_epochs:
                self.save_checkpoint(epoch, {
                    "train_loss": train_loss,
                    "val_loss": val_loss,
                    "qa_accuracy": qa_results["overall_accuracy"] if qa_results else None,
                })

        total_time = time.time() - start_time
        print(f"\nTraining complete in {total_time:.1f}s")

        # Diagnostic: flag the "low loss but useless output" failure mode.
        if self.metrics["val_loss"] and self.metrics["eval_accuracy"]:
            final_val = self.metrics["val_loss"][-1]
            final_acc = self.metrics["eval_accuracy"][-1]["overall_accuracy"]
            if final_val < 2.0 and final_acc < 0.5:
                print("\n  ⚠ LOW LOSS BUT USELESS OUTPUT")
                print(f"    val_loss={final_val:.3f} (low) yet exact-match acc={final_acc:.3f} (low).")
                print("    The model minimized cross-entropy without learning to answer.")

        # Save final results
        self.save_results()

        # Save samples to text file
        samples_path = self.output_dir / "samples.txt"
        with open(samples_path, 'w') as f:
            for sample in self.metrics["samples"]:
                mark = "CORRECT" if sample.get("correct") else "WRONG"
                f.write(f"\n--- Epoch {sample['epoch']} [{sample['task_type']}] {mark} ---\n")
                f.write(f"Prompt:\n{sample['prompt']}\n")
                f.write(f"Expected:  {sample['expected']}\n")
                f.write(f"Generated: {sample['generated']}\n")

        return self.metrics


In [ ]:
%%writefile bench.py
#!/usr/bin/env python3
"""
Reusable benchmark for the hybrid latent-state language model.

One entry point for everything:
  * Train one or more model variants.
  * Strict greedy exact-match eval (per task AND by difficulty bucket).
  * Print + save a rich report (loss curves, per-task acc, stratified acc,
    sample outputs, params, "low loss but useless output" flag).

Usage:
  # Fast CPU smoke of every model (tiny):
  python bench.py --quick

  # One model, more serious:
  python bench.py --models baseline --epochs 20 --n_samples 4000 --device cuda

  # Full battery on GPU:
  python bench.py --models baseline,latent_ssm,latent_ssm_decoder --epochs 30 --n_samples 8000 --device cuda

  # Just report existing experiment dirs (e.g. downloaded from Kaggle):
  python bench.py --analyze

The point of the strict, multi-dimensional eval is to catch the failure mode
where cross-entropy loss drops but the model still can't answer (low loss,
useless output). We never report a single number.
"""
import argparse
import json
import os
import random
import sys
import time
from pathlib import Path

import torch

sys.path.insert(0, str(Path(__file__).parent))
from src.dataset import (
    generate_dataset, format_for_training, build_prompt, parse_answer, _bucket,
)
from src.tokenizer import CharTokenizer
from src.models import BaselineTransformer, LatentSSM, LatentSSMDecoder
from src.trainer import Trainer, ExperimentConfig, TextDataset, DataLoader


# Model registry: key -> (model_type, latent_steps, think_every, tokens_per_step, d_model)
# `d_model` here is the per-spec override; latent models also use it as d_state.
# baseline_big is scaled to ~33M params to give a *same-size* autoregressive
# reference for the 34M latent models (the "first night win condition" needs
# an equal-parameter comparison, not 2M baseline vs 34M latent).
MODELS = {
    "baseline":             dict(model="baseline",           ls=0, te=0, tps=8, dm=256),
    "baseline_big":         dict(model="baseline_big",       ls=0, te=0, tps=8, dm=768),
    "latent_ssm":           dict(model="latent_ssm",         ls=0, te=0, tps=8, dm=256),
    "latent_ssm_think":     dict(model="latent_ssm",         ls=4, te=4, tps=8, dm=256),
    "latent_ssm_think8":    dict(model="latent_ssm",         ls=4, te=8, tps=8, dm=256),
    "latent_ssm_decoder":   dict(model="latent_ssm_decoder", ls=4, te=4, tps=8, dm=256),
}

QUICK = dict(d_model=48, n_samples=300, epochs=5, max_seq_len=384,
             location_max_chars=300, inventory_max_chars=300,
             transfer_max_chars=300, recall_max_chars=300, eval_every=2, batch_size=32)


def build_model(spec: dict, vocab_size: int, d_model: int):
    m = spec["model"]
    dm = spec.get("dm", d_model)
    if m == "baseline":
        return BaselineTransformer(vocab_size=vocab_size, d_model=dm, num_layers=4, nhead=8)
    if m == "baseline_big":
        return BaselineTransformer(vocab_size=vocab_size, d_model=dm, num_layers=6,
                                   nhead=12, dim_ff=2048)
    if m == "latent_ssm":
        return LatentSSM(vocab_size=vocab_size, d_state=dm, d_model=dm,
                         num_ssm_layers=2, latent_steps=spec["ls"], think_every=spec["te"])
    if m == "latent_ssm_decoder":
        return LatentSSMDecoder(vocab_size=vocab_size, d_state=dm, d_model=dm,
                                num_ssm_layers=2, latent_steps=spec["ls"],
                                tokens_per_step=spec["tps"], think_every=spec["te"])
    raise ValueError(m)


def run_one(key: str, args, dataset, device) -> dict:
    spec = MODELS[key]
    dm = spec.get("dm", args.d_model)
    random.seed(args.seed)
    torch.manual_seed(args.seed)

    print(f"STAGE: start exp={key} model={spec['model']} dm={dm} "
          f"steps={spec['ls']} think_every={spec['te']}")

    train_data = dataset[:int(0.8 * len(dataset))]
    val_data = dataset[int(0.8 * len(dataset)):]

    train_texts = [format_for_training(s) for s in train_data]
    val_texts = [format_for_training(s) for s in val_data]
    tokenizer = CharTokenizer(train_texts, max_vocab=256)

    model = build_model(spec, tokenizer.vocab_size, args.d_model).to(device)
    n_params = sum(p.numel() for p in model.parameters())

    config = ExperimentConfig(
        exp_id=key, model=spec["model"], d_model=dm,
        latent_steps=spec["ls"], batch_size=args.batch_size,
        num_epochs=args.epochs, max_seq_len=args.max_seq_len,
        eval_every=args.eval_every, seed=args.seed, device=str(device),
    )
    trainer = Trainer(model, config, tokenizer, output_dir=args.output_dir)
    train_loader = DataLoader(TextDataset(train_texts, tokenizer, max_len=args.max_seq_len),
                              batch_size=args.batch_size, shuffle=True)
    val_loader = DataLoader(TextDataset(val_texts, tokenizer, max_len=args.max_seq_len),
                            batch_size=args.batch_size)

    t0 = time.time()
    metrics = trainer.train(train_loader, val_loader, qa_dataset=val_data)
    elapsed = time.time() - t0

    qa = metrics["eval_accuracy"][-1] if metrics["eval_accuracy"] else None
    final_val = metrics["val_loss"][-1] if metrics["val_loss"] else None
    low_but_useless = bool(final_val is not None and final_val < 2.0
                           and qa is not None and qa["overall_accuracy"] < 0.5)

    # A few visible sample outputs (expected vs generated).
    samples = []
    for s in val_data[:6]:
        if not s.get("question"):
            continue
        prompt = build_prompt(s)
        gen = trainer.generate(prompt, max_tokens=20, temperature=0.0)
        samples.append(dict(task=s["task_type"], question=s["question"],
                            expected=s["answer"], generated=gen.strip()))

    return dict(
        key=key, model=spec["model"], latent_steps=spec["ls"], think_every=spec["te"],
        params=n_params, elapsed_s=elapsed,
        final_train_loss=metrics["train_loss"][-1] if metrics["train_loss"] else None,
        final_val_loss=final_val,
        overall_accuracy=qa["overall_accuracy"] if qa else None,
        task_accuracy=qa["task_accuracy"] if qa else None,
        stratified=qa["stratified"] if qa else None,
        n_eval=qa["n"] if qa else None,
        low_but_useless=low_but_useless,
        samples=samples,
    )
    print(f"STAGE: done exp={key} val_loss={final_val} acc={qa['overall_accuracy'] if qa else None}")


def render_report(results: list) -> str:
    _f = lambda x: f"{x:.3f}" if isinstance(x, (int, float)) else "—"
    lines = []
    lines.append("# Benchmark Report\n")
    lines.append(f"Models evaluated: {len(results)}\n")

    # Summary table
    lines.append("## Summary\n")
    lines.append("| model | steps | params | train_loss | val_loss | "
                 "exact_acc | low_loss_useless |")
    lines.append("|---|---|---|---|---|---|---|")
    for r in results:
        params = f"{r['params']:,}" if isinstance(r["params"], int) else str(r["params"])
        lines.append(
            f"| {r['key']} | {r['latent_steps']} | {params} | "
            f"{_f(r['final_train_loss'])} | {_f(r['final_val_loss'])} | "
            f"{_f(r['overall_accuracy'])} | {'YES' if r['low_but_useless'] else 'no'} |"
        )

    # Per-task accuracy
    lines.append("\n## Per-task exact-match accuracy\n")
    tasks = ["location", "inventory", "transfer", "recall"]
    lines.append("| model | " + " | ".join(tasks) + " |")
    lines.append("|---|" + "|".join(["---"] * len(tasks)) + "|")
    for r in results:
        ta = r["task_accuracy"] or {}
        lines.append("| " + r["key"] + " | " +
                     " | ".join(_f(ta.get(t)) for t in tasks) + " |")

    # Stratified
    lines.append("\n## Stratified by difficulty bucket\n")
    for r in results:
        if not r["stratified"]:
            continue
        lines.append(f"**{r['key']}**")
        for bkey, acc in sorted(r["stratified"].items()):
            lines.append(f"  - {bkey:<26} {_f(acc)}")
        lines.append("")

    # Sample outputs
    lines.append("\n## Sample outputs (expected vs generated)\n")
    for r in results:
        lines.append(f"**{r['key']}** (exact_acc={_f(r['overall_accuracy'])})")
        for s in r["samples"]:
            lines.append(f"  - [{s['task']}] Q: {s['question']}")
            lines.append(f"      expected: {s['expected']!r}")
            lines.append(f"      generated: {s['generated']!r}")
        lines.append("")

    return "\n".join(lines)


def analyze_existing(output_dir: str = "experiments") -> list:
    """Build a report from existing experiment dirs (no training)."""
    results = []
    for d in sorted(Path(output_dir).iterdir()):
        mfile = d / "metrics.json"
        if not mfile.exists():
            continue
        with open(mfile) as f:
            m = json.load(f)
        qa = m["eval_accuracy"][-1] if m.get("eval_accuracy") else None
        final_val = m["val_loss"][-1] if m.get("val_loss") else None
        samples = []
        stxt = d / "samples.txt"
        if stxt.exists():
            txt = stxt.read_text()
            for blk in txt.split("--- Epoch")[1:]:
                for line in blk.splitlines():
                    if line.strip().startswith("Expected:"):
                        samples.append(line.strip())
        results.append(dict(
            key=d.name, model=m.get("config", {}).get("model", "?"),
            latent_steps=m.get("config", {}).get("latent_steps", "?"),
            params="?", elapsed_s=0,
            final_train_loss=m["train_loss"][-1] if m.get("train_loss") else None,
            final_val_loss=final_val,
            overall_accuracy=qa["overall_accuracy"] if qa else None,
            task_accuracy=qa["task_accuracy"] if qa else None,
            stratified=qa["stratified"] if qa else None,
            n_eval=qa["n"] if qa else None,
            low_but_useless=bool(final_val is not None and final_val < 2.0
                                 and qa is not None and qa["overall_accuracy"] < 0.5),
            samples=[dict(task="", question="", expected=s, generated="") for s in samples[:6]],
        ))
    return results


def main():
    ap = argparse.ArgumentParser(description="Reusable latent-state benchmark")
    ap.add_argument("--models", default="baseline,baseline_big,latent_ssm,latent_ssm_think,latent_ssm_decoder",
                    help="Comma-separated model keys from the registry")
    ap.add_argument("--quick", action="store_true", help="Tiny CPU defaults")
    ap.add_argument("--epochs", type=int, default=20)
    ap.add_argument("--n_samples", type=int, default=5000)
    ap.add_argument("--d_model", type=int, default=256)
    ap.add_argument("--batch_size", type=int, default=32)
    ap.add_argument("--max_seq_len", type=int, default=768)
    ap.add_argument("--location_max_chars", type=int, default=600)
    ap.add_argument("--inventory_max_chars", type=int, default=600)
    ap.add_argument("--transfer_max_chars", type=int, default=600)
    ap.add_argument("--recall_max_chars", type=int, default=600)
    ap.add_argument("--eval_every", type=int, default=1)
    ap.add_argument("--seed", type=int, default=42)
    ap.add_argument("--device", default="auto")
    ap.add_argument("--output_dir", default="experiments")
    ap.add_argument("--analyze", action="store_true", help="Report existing dirs only")
    args = ap.parse_args()

    if args.analyze:
        results = analyze_existing(args.output_dir)
        report = render_report(results)
        print(report)
        Path("bench_report.md").write_text(report)
        print(f"\nSaved bench_report.md ({len(results)} experiments)")
        return

    if args.quick:
        for k, v in QUICK.items():
            setattr(args, k, v)

    device = torch.device("cuda" if (args.device == "auto" and torch.cuda.is_available())
                          else ("cpu" if args.device == "auto" else args.device))

    keys = [k.strip() for k in args.models.split(",") if k.strip()]
    print(f"Device: {device} | models: {keys}\n")
    print(f"STAGE: bench start models={keys} epochs={args.epochs} "
          f"n_samples={args.n_samples} device={device}")

    dataset = generate_dataset(
        n_samples=args.n_samples, seed=args.seed,
        location_max_chars=args.location_max_chars,
        inventory_max_chars=args.inventory_max_chars,
        transfer_max_chars=args.transfer_max_chars,
        recall_max_chars=args.recall_max_chars,
    )

    results = []
    for key in keys:
        if key not in MODELS:
            print(f"Unknown model key: {key} (known: {list(MODELS)})")
            continue
        print(f"\n##### {key} #####")
        try:
            results.append(run_one(key, args, dataset, device))
        except Exception as e:
            print(f"FAILED {key}: {e}")

    report = render_report(results)
    print("\n" + report)
    Path("bench_report.md").write_text(report)
    Path("bench_results.json").write_text(json.dumps(results, indent=2))
    print(f"\nSaved bench_report.md and bench_results.json ({len(results)} models)")
    print("STAGE: done")


if __name__ == "__main__":
    main()


In [ ]:
# Run the reusable benchmark (now on disk). It imports src/ and does train
# + strict exact-match QA eval + report. Output streams here so kaggle_run.py
# can watch STAGE: lines live. Outputs land in CWD (/kaggle/working) and are
# preserved as notebook outputs: experiments/<exp>/..., bench_report.md.
import sys, os

# Matched-parameter battery for the 'first night' win condition:
#   baseline      ~2M   (efficient AR reference)
#   baseline_big  ~33M  (AR baseline at the SAME size as the latent models)
#   latent_ssm       34M, thinking=0   (isolates the 'thinking' effect)
#   latent_ssm_think 34M, think every 4 (main hypothesis)
#   latent_ssm_decoder 34M, multi-token decoder
MODELS = 'baseline,baseline_big,latent_ssm,latent_ssm_think,latent_ssm_decoder'
cmd = (f"{sys.executable} bench.py --models {MODELS} "
       f"--epochs 20 --n_samples 5000 --device cuda --d_model 256 --eval_every 1")
print('STAGE: launch', cmd)
os.system(cmd)
print('NOTEBOOK DONE')


In [ ]:
# Show the generated report (also saved as bench_report.md).
report = Path('bench_report.md')
if report.exists():
    print(report.read_text())
else:
    print('bench_report.md not found (run may have failed).')
